# Disfluency-insertion analysis

Clean report on the held-out test set:

1. **Dataset stats** — the human reference set (paper's numbers shown for context).
2. **Reference-based metrics** — BLEU + BERTScore
3. **Token-level evaluation** — micro/macro precision-recall-F1 (+ accuracy, Jaccard, AUC)
4. **Type-specific accuracy** — typical (`<sil>`/filled-pause) vs atypical disfluencies.

Every table shows the **6 fine-tuned runs**, three **naive baselines**, and the **paper's
numbers** as a reference row. Set `DECODING` below to switch between greedy and the sampled
temperatures. For the sampled ones each metric is **averaged over the (3) sampled draws**.

In [1]:
# ============================================================================
# Setup: imports, paths, and which decoding to report.
# ============================================================================
import json, glob, difflib
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             jaccard_score, roc_auc_score)
from utils.swbd_disfluency_loader import Config, is_fragment, detok, _CLITICS, _PUNCT

pd.set_option("display.float_format", lambda v: f"{v:.2f}")

ROOT = Path("outputs/llm_runs_enriched")          # results directory

# Available decodings. "" = greedy (test_predictions.jsonl); the sampled files carry
# k=3 draws per row in the "predictions" column, which we AVERAGE over. Switch DECODING
# and re-run the notebook to report a different one.
DECODINGS = {"greedy": "", "T=0.5": "_sampled_t05", "T=1.0": "_sampled", "T=1.5": "_sampled_t15"}
DECODING  = "greedy"
SUFFIX    = DECODINGS[DECODING]
print(f"Reporting decoding = {DECODING!r}  (file suffix {SUFFIX!r})")

cfg = Config()
SIL_TOKEN, FILLER_WORDS, INSERTION_WORDS = cfg.sil_token, cfg.filler_words, cfg.insertion_words
TYPICAL_TYPES  = {"silent_pause", "filled_pause"}            # <sil>, uh, um, ...
ATYPICAL_TYPES = {"reparandum", "fragment", "insertion"}     # repeats, cut-offs, well/like/so

Reporting decoding = 'greedy'  (file suffix '')


In [2]:
# ============================================================================
# Tagging helpers, shared by every metric below.
#   tokenize        whitespace tokenizer; keeps "<sil>" atomic, lowercases, strips edge
#                   punctuation. Both sides of every comparison go through it.
#   tag_prediction  difflib(fluent, text) -> per-token 'O'/'D' via the paper's RULE
#                   (a token is disfluent if it differs from / is absent-from the fluent
#                   text). Applied to any string (human reference OR model output).
#   tag_reference   per-token 'O'/'D' from the GOLD disfl_labels (used only for the
#                   type-specific section, where exact typical/atypical types matter).
#   align_tag_seqs  align two tagged token sequences (difflib) -> aligned tag/type pairs.
#   get_draws       the k sampled predictions for a row (or [prediction] for greedy /
#                   baselines); _pred picks draw d -> lets every metric average over draws.
# ============================================================================
def tokenize(text):
    toks = []
    for t in (text or "").strip().split():
        if t == SIL_TOKEN:
            toks.append(t); continue
        c = t.strip('.,!?;:"()').lower()
        if c: toks.append(c)
    return toks

def tag_prediction(fluent_text, text):
    a, b = tokenize(fluent_text), tokenize(text)
    out = []
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(a=a, b=b, autojunk=False).get_opcodes():
        if op == "equal":
            out += [(b[k], "O", None) for k in range(j1, j2)]
        else:
            for k in range(j1, j2):
                tok = b[k]
                ttag = ("silent_pause" if tok == SIL_TOKEN else
                        "filled_pause" if tok in FILLER_WORDS else
                        "insertion"    if tok in INSERTION_WORDS else
                        "fragment"     if is_fragment(tok) else "other_disfl")
                out.append((tok, "D", ttag))
    return out

def tag_reference(tokens, labels):
    return [(t.lower(), "O", None) if l is None else (t.lower(), "D", l)
            for t, l in zip(tokens, labels)]

def align_tag_seqs(ref_seq, pred_seq):
    rt, pt = [x[0] for x in ref_seq], [x[0] for x in pred_seq]
    pairs = []
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(a=rt, b=pt, autojunk=False).get_opcodes():
        if op == "equal":
            for d in range(i2 - i1):
                r, p = ref_seq[i1+d], pred_seq[j1+d]; pairs.append((r[1], p[1], r[2], p[2]))
        elif op == "replace":
            n = min(i2 - i1, j2 - j1)
            for d in range(n):
                r, p = ref_seq[i1+d], pred_seq[j1+d]; pairs.append((r[1], p[1], r[2], p[2]))
            for d in range(n, i2-i1): pairs.append((ref_seq[i1+d][1], "O", ref_seq[i1+d][2], None))
            for d in range(n, j2-j1): pairs.append(("O", pred_seq[j1+d][1], None, pred_seq[j1+d][2]))
        elif op == "delete":
            for d in range(i2-i1): pairs.append((ref_seq[i1+d][1], "O", ref_seq[i1+d][2], None))
        elif op == "insert":
            for d in range(j2-j1): pairs.append(("O", pred_seq[j1+d][1], None, pred_seq[j1+d][2]))
    return pairs

def get_draws(row):
    preds = getattr(row, "predictions", None)
    if isinstance(preds, list) and preds:
        return [p or "" for p in preds]
    return [getattr(row, "prediction", "") or ""]

def _pred(row, d):
    dr = get_draws(row)
    return dr[d] if d < len(dr) else dr[-1]


def gold_tags_ws(tokens, labels):
    """Rebuild the GOLD disfluent tokens into the SAME whitespace tokenization used on the
    prediction side (reusing the loader's clitic/punct merge rules), carrying labels: a merged
    token (e.g. "may"+"'ve" -> "may've") is disfluent if ANY of its parts was. Removes the
    PTB-vs-whitespace mismatch so the type-specific alignment is clean, keeping exact gold types."""
    surf, labs = [], []
    for tok, lab in zip(tokens, labels):
        if surf and (tok.lower() in _CLITICS or tok in _PUNCT or tok.startswith("'")):
            surf[-1] = surf[-1] + tok
            if labs[-1] is None:
                labs[-1] = lab                     # disfluency wins on merge
        else:
            surf.append(tok); labs.append(lab)
    out_t, out_l = [], []
    for s, l in zip(surf, labs):
        for t in tokenize(s):                      # strip punct / lowercase / keep <sil>
            out_t.append(t); out_l.append(l)
    return out_t, out_l

def tag_reference_ws(tokens, labels):
    """Like tag_reference, but on the whitespace-consistent gold tokens (see gold_tags_ws)."""
    toks, labs = gold_tags_ws(tokens, labels)
    return [(t, "O", None) if l is None else (t, "D", l) for t, l in zip(toks, labs)]

In [3]:
# ============================================================================
# Naive baselines, added as extra "run"s so every table includes them:
#   baseline_copy            insert nothing (output the fluent sentence)      -> the floor.
#   baseline_random          the human's exact disfluency tokens scattered to RANDOM
#                            positions (token-level). Matches the human count & types but
#                            not placement -> isolates placement skill.
#   baseline_random_typical  same, but only the simple (<sil>/filled-pause) disfluencies
#                            the models actually produce (span-preserving).
# Gold labels are used only to BUILD the baselines; they are scored by the same heuristic
# as the models, so the comparison is fair.
# ============================================================================
def add_baselines(df):
    import random
    rng = random.Random(42)
    df = df[~df["run"].astype(str).str.startswith("baseline_")].copy()
    gold = {}
    with open(ROOT / "splits" / "test.jsonl") as fh:
        for i, line in enumerate(fh):
            r = json.loads(line); gold[(r["conv"], i)] = (r["disfluent_tokens"], r["disfl_labels"])
    TYP = {"silent_pause", "filled_pause"}
    uniq = df.drop_duplicates(subset=["conv", "id"]).copy()
    copy, rand, typ = uniq.copy(), uniq.copy(), uniq.copy()
    for d, name in ((copy, "baseline_copy"), (rand, "baseline_random"), (typ, "baseline_random_typical")):
        d["run"] = name
        for col in ("model_key", "model_id", "context_mode"):
            if col in d.columns: d[col] = name
        if "predictions" in d.columns: d["predictions"] = None   # baselines are a single draw
    copy["prediction"] = copy["fluent"]

    def scatter(conv, id_, fluent, as_spans, only=None):
        g = gold.get((conv, id_))
        if not g: return fluent
        toks, labs = g
        keep = lambda l: (l is not None) and (only is None or l in only)
        if as_spans:                                   # keep contiguous disfluency spans intact
            units, cur = [], []
            for t, l in zip(toks, labs):
                if keep(l): cur.append(t)
                elif cur: units.append(cur); cur = []
            if cur: units.append(cur)
        else:                                          # scatter each disfluency token independently
            units = [[t] for t, l in zip(toks, labs) if keep(l)]
        out = fluent.split()
        for u in units:
            pos = rng.randint(0, len(out)); out[pos:pos] = u    # insert the unit at a random gap
        return " ".join(out)

    rand["prediction"] = [scatter(c, i, f, False)                for c, i, f in zip(rand["conv"], rand["id"], rand["fluent"])]
    typ["prediction"]  = [scatter(c, i, f, True, only=TYP)       for c, i, f in zip(typ["conv"], typ["id"], typ["fluent"])]
    return pd.concat([df, copy, rand, typ], ignore_index=True)

In [4]:
# ============================================================================
# Load the selected decoding's predictions, attach gold token labels, add baselines.
# ============================================================================
def load_predictions(suffix):
    rows = []
    for f in sorted(glob.glob(str(ROOT / f"*/*/test_predictions{suffix}.jsonl"))):
        rows += [json.loads(l) for l in open(f) if l.strip()]
    d = pd.DataFrame(rows)
    d["run"] = d["model_key"] + " / " + d["context_mode"]
    return d

gold_df = (pd.DataFrame([json.loads(l) for l in open(ROOT / "splits" / "test.jsonl")])
           .reset_index().rename(columns={"index": "id"}))

df = load_predictions(SUFFIX)
df = df.merge(gold_df[["conv", "id", "disfluent_tokens", "disfl_labels"]], on=["conv", "id"], how="left")
df = add_baselines(df)

MODEL_RUNS    = sorted(r for r in df["run"].unique() if not r.startswith("baseline_"))
BASELINE_RUNS = ["baseline_copy", "baseline_random", "baseline_random_typical"]
RUN_ORDER     = MODEL_RUNS + BASELINE_RUNS
print(f"{len(df):,} rows | decoding={DECODING} | runs:")
for r in RUN_ORDER: print("  ", r)

16,038 rows | decoding=greedy | runs:
   G4-4EB-it / no_context
   G4-4EB-it / with_context
   L2-7B-Chat / no_context
   L2-7B-Chat / with_context
   L3.1-8B-Instruct / no_context
   L3.1-8B-Instruct / with_context
   baseline_copy
   baseline_random
   baseline_random_typical


In [5]:
# Row counts per model_key x context_mode combo — make sure nothing is missing/duplicated
display(df.groupby(["model_key", "context_mode"]).size().rename("n_rows"))

# Check for nulls in the columns we actually need
display(df[["fluent", "reference", "prediction"]].isna().sum())

model_key                context_mode           
G4-4EB-it                no_context                 1782
                         with_context               1782
L2-7B-Chat               no_context                 1782
                         with_context               1782
L3.1-8B-Instruct         no_context                 1782
                         with_context               1782
baseline_copy            baseline_copy              1782
baseline_random          baseline_random            1782
baseline_random_typical  baseline_random_typical    1782
Name: n_rows, dtype: int64

fluent        0
reference     0
prediction    0
dtype: int64

## 1. Dataset stats
The human (reference) test set. 
Paper's full-set stats: train/test = 32,490 / 3,610 sentences, ~24% fluent / ~33% disfluent tokens, disfluency rate ~24%.

In [6]:
# Stats computed from the gold labels of the unique test sentences.
uniq = df.drop_duplicates(subset=["conv", "id"])
n_dis = n_tok_dis = n_tok_flu = n_typ = n_aty = 0
per_sent = []
for r in uniq.itertuples():
    labs = r.disfl_labels
    d = sum(1 for l in labs if l is not None); per_sent.append(d)
    n_dis += d; n_tok_dis += len(labs); n_tok_flu += sum(1 for l in labs if l is None)
    n_typ += sum(1 for l in labs if l in TYPICAL_TYPES)
    n_aty += sum(1 for l in labs if l in ATYPICAL_TYPES)

pd.Series({
    "n_sentences":               len(uniq),
    "avg_tokens_fluent":         n_tok_flu / len(uniq),
    "avg_tokens_disfluent":      n_tok_dis / len(uniq),
    "disfluency_rate_%":         100 * n_dis / n_tok_dis,
    "disfluencies_per_sentence": np.mean(per_sent),
    "%_typical":                 100 * n_typ / max(n_typ + n_aty, 1),
    "%_atypical":                100 * n_aty / max(n_typ + n_aty, 1),
}, name="our_test_set").to_frame().round(2)

,our_test_set
n_sentences,1782.00
avg_tokens_fluent,16.27
avg_tokens_disfluent,20.87
disfluency_rate_%,22.07
disfluencies_per_sentence,4.61
%_typical,69.84
%_atypical,30.16


## 2. Reference-based metrics (BLEU / BERTScore)
Both are averaged over the sampled draws; BERTScore is raw (`rescale_with_baseline=False`), shown as a percentage. *(This cell loads RoBERTa and can take a few minutes.)*

In [7]:
import sacrebleu
from bert_score import BERTScorer
_scorer = BERTScorer(lang="en", rescale_with_baseline=False)

def bleu_bert(g):
    rows_ = list(g.itertuples()); k = len(get_draws(rows_[0]))
    refs = [r.reference for r in rows_]
    bl, bp, br, bf = [], [], [], []
    for d in range(k):
        preds = [_pred(r, d) for r in rows_]
        bl.append(sacrebleu.corpus_bleu(preds, [refs]).score)
        P, R, F = _scorer.score(preds, refs, batch_size=256)
        bp.append(100 * P.mean().item()); br.append(100 * R.mean().item()); bf.append(100 * F.mean().item())
    return dict(BLEU=np.mean(bl), BERT_P=np.mean(bp), BERT_R=np.mean(br), BERT_F1=np.mean(bf))

ref_tab = pd.DataFrame({run: bleu_bert(df[df.run == run]) for run in RUN_ORDER}).T
ref_tab.loc["— paper: llama-2-7b"]    = {"BLEU": 55.24, "BERT_P": 92.87, "BERT_R": 93.70, "BERT_F1": 93.27}
ref_tab.loc["— paper: flan-t5-large"] = {"BLEU": 44.40, "BERT_P": 96.24, "BERT_R": 91.68, "BERT_F1": 93.86}
ref_tab.round(2)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,BLEU,BERT_P,BERT_R,BERT_F1
G4-4EB-it / no_context,67.11,96.90,95.08,95.96
G4-4EB-it / with_context,67.56,96.90,95.14,96.00
L2-7B-Chat / no_context,67.61,96.89,95.15,96.00
L2-7B-Chat / with_context,67.52,96.85,95.14,95.97
L3.1-8B-Instruct / no_context,68.00,96.88,95.20,96.01
L3.1-8B-Instruct / with_context,68.32,96.92,95.27,96.07
baseline_copy,44.15,97.53,89.71,93.42
baseline_random,55.61,94.88,95.17,95.02
baseline_random_typical,63.82,96.16,95.70,95.93
— paper: llama-2-7b,55.24,92.87,93.70,93.27


## 3. Token-level evaluation
The paper's rule-based tagger applied to **both** the human reference and the model output (`tag_prediction`), then aligned. **micro** = pooled over all tokens; **macro** = mean of per-sentence scores (the paper's F1 is not harmonic(P,R), so it most likely reports macro, but idk, filled macro/micro with same values, the do not specify). Averaged over the sampled draws.

In [8]:
def token_scores(g):
    rows_ = list(g.itertuples()); k = len(get_draws(rows_[0]))
    acc = defaultdict(list)
    for d in range(k):
        rb, pb, mp, mr, mf = [], [], [], [], []
        for r in rows_:
            pairs = align_tag_seqs(tag_prediction(r.fluent, r.reference),
                                   tag_prediction(r.fluent, _pred(r, d)))
            a = [1 if x[0] == "D" else 0 for x in pairs]
            b = [1 if x[1] == "D" else 0 for x in pairs]
            rb += a; pb += b
            if sum(a) or sum(b):
                p_, r_, f_, _ = precision_recall_fscore_support(a, b, average="binary", zero_division=0)
                mp.append(p_); mr.append(r_); mf.append(f_)
        P, R, F, _ = precision_recall_fscore_support(rb, pb, average="binary", zero_division=0)
        acc["P_micro"].append(100 * P);  acc["R_micro"].append(100 * R);  acc["F1_micro"].append(100 * F)
        acc["accuracy"].append(100 * accuracy_score(rb, pb))
        acc["jaccard"].append(100 * jaccard_score(rb, pb, zero_division=0))
        acc["auc"].append(roc_auc_score(rb, pb) if len(set(rb)) > 1 else np.nan)
        acc["P_macro"].append(100 * np.mean(mp)); acc["R_macro"].append(100 * np.mean(mr)); acc["F1_macro"].append(100 * np.mean(mf))
    return {key: np.mean(v) for key, v in acc.items()}

COLS = ["P_micro", "R_micro", "F1_micro", "accuracy", "jaccard", "auc", "P_macro", "R_macro", "F1_macro"]
tok_tab = pd.DataFrame({run: token_scores(df[df.run == run]) for run in RUN_ORDER}).T[COLS]
tok_tab.loc["— paper"] = {"P_micro": 52.2, "R_micro": 85.8, "F1_micro": 61.2, "accuracy": 81.6,
                                    "jaccard": 30.4, "auc": 0.603, "P_macro": 52.2, "R_macro": 85.8, "F1_macro": 61.2}
tok_tab.round(2)

,P_micro,R_micro,F1_micro,accuracy,jaccard,auc,P_macro,R_macro,F1_macro
G4-4EB-it / no_context,67.05,25.20,36.63,80.97,22.42,0.61,62.26,35.23,40.40
G4-4EB-it / with_context,65.77,25.98,37.24,80.93,22.88,0.61,62.36,35.94,40.87
L2-7B-Chat / no_context,64.90,26.02,37.14,80.84,22.81,0.61,62.48,35.84,40.87
L2-7B-Chat / with_context,61.13,26.16,36.64,80.41,22.43,0.61,61.99,35.79,40.68
L3.1-8B-Instruct / no_context,62.88,26.46,37.25,80.67,22.89,0.61,62.04,36.29,41.03
L3.1-8B-Instruct / with_context,65.62,27.39,38.65,81.10,23.95,0.62,62.60,37.44,41.95
baseline_copy,0.00,0.00,0.00,77.55,0.00,0.50,0.00,0.00,0.00
baseline_random,26.70,27.22,26.96,72.16,15.58,0.55,23.07,23.37,23.21
baseline_random_typical,35.37,25.18,29.42,75.89,17.25,0.57,28.66,21.72,24.13
— paper,52.20,85.80,61.20,81.60,30.40,0.60,52.20,85.80,61.20


## 4. Type-specific accuracy (typical vs atypical)
As in the paper (Table 5). The **reference** uses the GOLD `disfl_labels` on the same whitespace tokenization as the prediction (`gold_tags_ws`). Shown as two tables - **micro** (token-pooled) and **macro** (per-sentence mean, skipping sentences with none of that type; more stable for the rare atypical class). The paper's numbers are probably macro? Shown in **both** tables anyway. Averaged over the sampled draws.

In [ ]:
def bucket(t):
    if t in TYPICAL_TYPES: return "typical"
    if t in ATYPICAL_TYPES or t == "other_disfl": return "atypical"
    return None

def _mean(xs):                       # mean ignoring NaNs; NaN if all-NaN (no RuntimeWarning)
    xs = [v for v in xs if not np.isnan(v)]
    return np.mean(xs) if xs else np.nan

def type_scores(g):
    """Per-type recall / precision, both MICRO (token-pooled over the corpus) and MACRO (mean of
    per-sentence scores, skipping sentences with none of that type). Averaged over the sampled
    draws. Reference uses gold types on whitespace-consistent tokens (tag_reference_ws)."""
    rows_ = list(g.itertuples()); k = len(get_draws(rows_[0]))
    out = {}
    for b in ("typical", "atypical"):
        mic_r, mic_p, mac_r, mac_p = [], [], [], []
        for d in range(k):
            RH = RT = PH = PT = 0
            srec, sprec = [], []
            for r in rows_:
                pairs = align_tag_seqs(tag_reference_ws(r.disfluent_tokens, r.disfl_labels),
                                       tag_prediction(r.fluent, _pred(r, d)))
                rh = rt = ph = pt = 0
                for rtag, ptag, rty, pty in pairs:
                    if rtag == "D" and bucket(rty) == b: rt += 1; rh += (ptag == "D")
                    if ptag == "D" and bucket(pty) == b: pt += 1; ph += (rtag == "D")
                RH += rh; RT += rt; PH += ph; PT += pt
                if rt: srec.append(rh / rt)
                if pt: sprec.append(ph / pt)
            mic_r.append(100 * RH / RT if RT else np.nan)
            mic_p.append(100 * PH / PT if PT else np.nan)
            mac_r.append(100 * np.mean(srec) if srec else np.nan)
            mac_p.append(100 * np.mean(sprec) if sprec else np.nan)
        out[f"{b}_recall_micro"]    = _mean(mic_r)
        out[f"{b}_precision_micro"] = _mean(mic_p)
        out[f"{b}_recall_macro"]    = _mean(mac_r)
        out[f"{b}_precision_macro"] = _mean(mac_p)
    return out

_ts = pd.DataFrame({run: type_scores(df[df.run == run]) for run in RUN_ORDER}).T

def type_view(kind):                 # kind in {"micro", "macro"} -> a compact 4-column table
    v = _ts[[f"typical_recall_{kind}", f"typical_precision_{kind}",
             f"atypical_recall_{kind}", f"atypical_precision_{kind}"]].copy()
    v.columns = ["typical_recall", "typical_precision", "atypical_recall", "atypical_precision"]

    v.loc["— paper: SIL (typical)"]    = {"typical_recall": 93.4, "typical_precision": 49.5}
    v.loc["— paper: DISFL (atypical)"] = {"atypical_recall": 82.5, "atypical_precision": 53.4}
    return v.round(2)

print("Type-specific - MICRO (token-pooled):")
display(type_view("micro"))
print("\nType-specific - MACRO (per-sentence mean; the paper's protocol):")
display(type_view("macro"))

/tmp/ipykernel_1225042/1157489313.py:21: RuntimeWarning: Mean of empty slice
  out[f"{b}_recall"] = np.nanmean(recs); out[f"{b}_precision"] = np.nanmean(precs)


,typical_recall,typical_precision,atypical_recall,atypical_precision
G4-4EB-it / no_context,34.53,67.75,1.99,29.03
G4-4EB-it / with_context,35.21,67.11,3.15,30.17
L2-7B-Chat / no_context,35.39,66.00,2.72,24.73
L2-7B-Chat / with_context,35.46,63.57,2.81,17.00
L3.1-8B-Instruct / no_context,36.01,65.41,2.72,12.50
L3.1-8B-Instruct / with_context,36.97,66.89,3.59,31.54
baseline_copy,0.00,NaN,0.00,NaN
baseline_random,31.22,30.28,16.51,17.44
baseline_random_typical,34.16,35.44,3.15,NaN
— paper: SIL (typical),93.40,49.50,NaN,NaN
